# 近期窗口融合实验

本 Notebook 完整执行全历史模型、近期 1702 期专家、验证融合与测试预测。


## Setup

### 1. 参数与运行模式

In [ ]:
from __future__ import annotations
import gc, hashlib, json, os, random, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data.z").exists() and (candidate / "02_experiments").exists():
            return candidate
    raise RuntimeError("无法定位项目根目录：请从项目目录或实验目录启动 Notebook。")

PROJECT_ROOT = find_project_root()
EXPERIMENT_ID = "exp_007_recent_window_blend"
DATASET_DIR = PROJECT_ROOT / "03_cache" / "processed_data_v1"
OUTPUT_DIR = PROJECT_ROOT / "04_results" / EXPERIMENT_ID
MANIFEST_PATH = DATASET_DIR / "manifest.json"
READY_PATH = DATASET_DIR / "READY"
SEED = 42
ROUNDS_TO_SCORE = tuple(int(value) for value in "8".split(","))
RUN_SCREENING = True
SCREEN_PROFILE = "blend_refit"
PARAM_PROFILE = "tuned_public_best"
TRAIN_STOCK_CAP = 1200
NUM_THREADS = max(1, (os.cpu_count() or 8) - 2)
RUN_FINAL_CANDIDATE = True
random.seed(SEED); np.random.seed(SEED)
print({'dataset': str(DATASET_DIR), 'screening': RUN_SCREENING,
       'profile': SCREEN_PROFILE, 'params': PARAM_PROFILE,
       'stock_cap': TRAIN_STOCK_CAP, 'threads': NUM_THREADS,
       'final_candidate': RUN_FINAL_CANDIDATE})

## Data

### 2. 验证 READY 并加载固定视图

In [ ]:
def sha256(path, block_size=16 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while block := handle.read(block_size):
            digest.update(block)
    return digest.hexdigest()

if not READY_PATH.exists() or not MANIFEST_PATH.exists():
    raise RuntimeError('processed_data_v1 没有 READY，禁止训练')
ready = json.loads(READY_PATH.read_text(encoding='utf-8'))
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
assert manifest['status'] == 'ready'
assert sha256(MANIFEST_PATH) == ready['manifest_sha256']

def load_common(split):
    directory = DATASET_DIR / 'common'
    result = {
        'time': np.load(directory / f'{split}_time.npy', mmap_mode='r'),
        'stock': np.load(directory / f'{split}_stock.npy', mmap_mode='r'),
        'groups': np.load(directory / f'{split}_group_sizes.npy', mmap_mode='r'),
    }
    if split != 'test':
        result['y'] = np.load(directory / f'{split}_y.npy', mmap_mode='r')
        result['relevance'] = np.load(directory / f'{split}_relevance.npy', mmap_mode='r')
    return result

def load_tree(split):
    matrix = np.load(DATASET_DIR / 'tree' / f'{split}_X.npy', mmap_mode='r')
    assert matrix.shape == (manifest['expected_rows'][split], 419)
    return matrix

common = {split: load_common(split) for split in ('train', 'valid', 'test')}
tree = {split: load_tree(split) for split in ('train', 'valid', 'test')}
summary = pd.DataFrame([
    {'split': split, 'rows': values['time'].size,
     'time_start': int(values['time'][0]), 'time_stop': int(values['time'][-1]) + 1,
     'time_points': values['groups'].size, 'group_sum': int(values['groups'].sum())}
    for split, values in common.items()
])
assert np.isfinite(tree['train'][:32]).all()
display(summary)

## Context & Methods

### 3. 特征集、时间窗口与统一 RankIC

In [ ]:
FEATURE_SETS = {
    'legacy_328': {'stop': 328, 'categorical': []},
    'numeric_408': {'stop': 408, 'categorical': []},
    'full_419': {'stop': 419, 'categorical': list(range(408, 417))},
}
TRAIN_START, TRAIN_STOP = 486, 2918
VALID_START, VALID_STOP = 2918, 3161

def rank_ic(prediction, target):
    usable = np.isfinite(prediction) & np.isfinite(target)
    if usable.sum() < 2:
        return np.nan
    x, y = rankdata(prediction[usable]), rankdata(target[usable])
    if x.std() == 0 or y.std() == 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])

def group_rank_ic_series(prediction, target, groups):
    values, offset = [], 0
    for size in groups:
        size = int(size)
        values.append(rank_ic(prediction[offset:offset + size], target[offset:offset + size]))
        offset += size
    assert offset == prediction.size
    return np.asarray(values, dtype=np.float64)

def group_rank_transform(prediction, groups):
    ranked = np.empty(prediction.size, dtype=np.float32)
    offset = 0
    for size in groups:
        size = int(size)
        ranked[offset:offset + size] = rankdata(prediction[offset:offset + size]).astype(np.float32) / size
        offset += size
    assert offset == prediction.size
    return ranked

def row_slice_for_times(split, start_time, stop_time):
    times = common[split]['time']
    start = int(np.searchsorted(times, start_time, side='left'))
    stop = int(np.searchsorted(times, stop_time, side='left'))
    split_start = int(times[0])
    group_start = start_time - split_start
    group_stop = stop_time - split_start
    groups = np.asarray(common[split]['groups'][group_start:group_stop], dtype=np.int32)
    assert int(groups.sum()) == stop - start
    return slice(start, stop), groups

assert abs(rank_ic(np.arange(10), np.arange(10)) - 1.0) < 1e-12
print({name: spec['stop'] for name, spec in FEATURE_SETS.items()})

## Steps

### 4. 第一轮 LightGBM 筛选

`quick` 只比较 328 与 408 列；`extended` 增加 1216/730 期近期窗口；`categories` 再加入 419 列类别消融。快速筛选训练到 32 轮，并同时读取 8/16/32 轮预测；晋级后再扩大轮数。

In [ ]:
PROFILE_CONFIGS = {
    'quick': [
        {'name': 'legacy_328_full', 'features': 'legacy_328', 'window': None},
        {'name': 'numeric_408_full', 'features': 'numeric_408', 'window': None},
    ],
    'windows': [
        {'name': 'legacy_328_recent1945', 'features': 'legacy_328', 'window': 1945},
        {'name': 'legacy_328_recent1216', 'features': 'legacy_328', 'window': 1216},
        {'name': 'legacy_328_recent730', 'features': 'legacy_328', 'window': 730},
    ],
    'windows_refine': [
        {'name': 'legacy_328_recent2188', 'features': 'legacy_328', 'window': 2188},
        {'name': 'legacy_328_recent1702', 'features': 'legacy_328', 'window': 1702},
    ],
    'blend_refit': [
        {'name': 'legacy_328_full', 'features': 'legacy_328', 'window': None},
        {'name': 'legacy_328_recent1702', 'features': 'legacy_328', 'window': 1702},
    ],
    'categories': [
        {'name': 'full_419_full', 'features': 'full_419', 'window': None},
    ],
}
if SCREEN_PROFILE == 'feature_screen':
    SCREEN_CONFIGS = PROFILE_CONFIGS['quick'] + PROFILE_CONFIGS['categories']
elif SCREEN_PROFILE == 'window_screen':
    SCREEN_CONFIGS = PROFILE_CONFIGS['windows'] + PROFILE_CONFIGS['windows_refine']
elif SCREEN_PROFILE in PROFILE_CONFIGS:
    SCREEN_CONFIGS = PROFILE_CONFIGS[SCREEN_PROFILE]
else:
    raise ValueError(f'未知实验配置：{SCREEN_PROFILE}')

LGB_PARAMS = {
    'objective': 'lambdarank', 'metric': 'None', 'learning_rate': 0.03,
    'num_leaves': 63, 'min_data_in_leaf': 300, 'feature_fraction': 0.80,
    'bagging_fraction': 0.80, 'bagging_freq': 1, 'lambda_l1': 0.1,
    'lambda_l2': 1.0, 'max_bin': 127, 'label_gain': list(range(64)),
    'lambdarank_truncation_level': 1024, 'verbosity': -1,
    'seed': SEED, 'feature_fraction_seed': SEED, 'bagging_seed': SEED,
    'num_threads': NUM_THREADS,
}
if PARAM_PROFILE == 'tuned_public_best':
    LGB_PARAMS.update({
        'learning_rate': 0.0228695, 'num_leaves': 79, 'min_data_in_leaf': 147,
        'feature_fraction': 0.80936, 'bagging_fraction': 0.647764,
        'lambda_l1': 2.35724, 'lambda_l2': 0.238705, 'max_bin': 127,
    })
elif PARAM_PROFILE != 'base':
    raise ValueError(f'未知 Y1_PARAM_PROFILE={PARAM_PROFILE}')
display(pd.DataFrame(SCREEN_CONFIGS))

In [ ]:
def score_prediction(prediction, target, groups):
    per_time = group_rank_ic_series(prediction, target, groups)
    quarters = np.array_split(per_time, 4)
    result = {
        'mean_rank_ic': float(np.nanmean(per_time)),
        'std_rank_ic': float(np.nanstd(per_time)),
        'late_half_rank_ic': float(np.nanmean(per_time[len(per_time) // 2:])),
        'worst_quarter_rank_ic': float(min(np.nanmean(part) for part in quarters)),
        'negative_time_share': float(np.mean(per_time < 0)),
    }
    result['selection_score'] = (0.50 * result['mean_rank_ic']
                                 + 0.30 * result['late_half_rank_ic']
                                 + 0.20 * result['worst_quarter_rank_ic'])
    return result

def train_screen_config(config):
    import lightgbm as lgb
    feature_spec = FEATURE_SETS[config['features']]
    feature_stop = feature_spec['stop']
    train_start = TRAIN_START if config['window'] is None else max(TRAIN_START, TRAIN_STOP - config['window'])
    train_rows, full_train_groups = row_slice_for_times('train', train_start, TRAIN_STOP)
    if TRAIN_STOCK_CAP > 0:
        capped_groups = np.minimum(full_train_groups, TRAIN_STOCK_CAP).astype(np.int32)
        row_indices = np.empty(int(capped_groups.sum()), dtype=np.int64)
        source_offset, target_offset = int(train_rows.start), 0
        for full_size, capped_size in zip(full_train_groups, capped_groups):
            positions = np.linspace(0, int(full_size) - 1, int(capped_size), dtype=np.int64)
            row_indices[target_offset:target_offset + int(capped_size)] = source_offset + positions
            source_offset += int(full_size); target_offset += int(capped_size)
        assert source_offset == int(train_rows.stop) and target_offset == row_indices.size
        X_train = np.asarray(tree['train'][row_indices, :feature_stop], dtype=np.float32)
        y_train = np.asarray(common['train']['relevance'][row_indices], dtype=np.int8)
        train_groups = capped_groups
    else:
        X_train = tree['train'][train_rows, :feature_stop]
        y_train = np.asarray(common['train']['relevance'][train_rows], dtype=np.int8)
        train_groups = full_train_groups
    dataset = lgb.Dataset(
        X_train, label=y_train, group=train_groups,
        categorical_feature=feature_spec['categorical'] or 'auto',
        free_raw_data=True,
    )
    started = time.time()
    model = lgb.train(LGB_PARAMS, dataset, num_boost_round=max(ROUNDS_TO_SCORE),
                      callbacks=[lgb.log_evaluation(0)])
    elapsed = time.time() - started
    X_valid = tree['valid'][:, :feature_stop]
    y_valid = np.asarray(common['valid']['y'], dtype=np.float32)
    valid_groups = np.asarray(common['valid']['groups'], dtype=np.int32)
    rows, predictions = [], {}
    for rounds in ROUNDS_TO_SCORE:
        prediction = model.predict(X_valid, num_iteration=rounds).astype(np.float32)
        key = f"{PARAM_PROFILE}_cap{TRAIN_STOCK_CAP}__{config['name']}__r{rounds}"
        predictions[key] = prediction
        rows.append({
            'key': key, 'param_profile': PARAM_PROFILE,
            'train_stock_cap': TRAIN_STOCK_CAP, 'num_threads': NUM_THREADS,
            'config': config['name'], 'features': config['features'],
            'feature_count': feature_stop, 'window': config['window'] or 'full',
            'train_start': train_start, 'train_stop': TRAIN_STOP,
            'train_rows': X_train.shape[0], 'rounds': rounds,
            'training_seconds': elapsed,
            **score_prediction(prediction, y_valid, valid_groups),
        })
    del dataset, X_train, y_train
    gc.collect()
    return model, rows, predictions

print('训练函数已就绪，本实验默认执行。')

In [ ]:
result_path = OUTPUT_DIR / 'screening_results.csv'
screening_results = pd.read_csv(result_path) if result_path.exists() else pd.DataFrame()
screening_predictions = {}
screening_models = {}
if RUN_SCREENING:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    all_rows = []
    for config in SCREEN_CONFIGS:
        print(f"开始 {config['name']} ...", flush=True)
        model, rows, predictions = train_screen_config(config)
        screening_models[config['name']] = model
        screening_predictions.update(predictions)
        all_rows.extend(rows)
        current = pd.DataFrame(rows).sort_values('selection_score', ascending=False)
        display(current[['key', 'mean_rank_ic', 'late_half_rank_ic',
                         'worst_quarter_rank_ic', 'selection_score']])
    new_results = pd.DataFrame(all_rows)
    screening_results = pd.concat([screening_results, new_results], ignore_index=True)
    screening_results = screening_results.drop_duplicates('key', keep='last').sort_values('selection_score', ascending=False)
    screening_results.to_csv(OUTPUT_DIR / 'screening_results.csv', index=False, encoding='utf-8-sig')
    display(screening_results)
else:
    if result_path.exists():
        screening_results = pd.read_csv(result_path)
        display(screening_results.head(10))
    else:
        print('快速检查模式：数据接口通过，尚未运行模型筛选。')

## Results

### 5. 互补性与 Rank 融合诊断

只有本次实际训练时才运行。融合使用逐时点排名，避免不同模型预测尺度造成伪权重。

In [ ]:
blend_path = OUTPUT_DIR / 'screening_blends.csv'
blend_results = pd.read_csv(blend_path) if blend_path.exists() else pd.DataFrame()
if RUN_SCREENING and not screening_results.empty:
    legacy_candidates = screening_results[
        (screening_results['config'] == 'legacy_328_full')
        & screening_results['key'].isin(screening_predictions)
    ]
    legacy_row = legacy_candidates.iloc[0] if not legacy_candidates.empty else None
    alternatives = screening_results[
        (screening_results['config'] != 'legacy_328_full')
        & screening_results['key'].isin(screening_predictions)
    ]
    if (legacy_row is not None and not alternatives.empty and legacy_row['key'] in screening_predictions
            and alternatives.iloc[0]['key'] in screening_predictions):
        alternative_row = alternatives.iloc[0]
        legacy_rank = group_rank_transform(screening_predictions[legacy_row['key']], common['valid']['groups'])
        alternative_rank = group_rank_transform(screening_predictions[alternative_row['key']], common['valid']['groups'])
        prediction_rank_corr = float(np.corrcoef(legacy_rank, alternative_rank)[0, 1])
        blend_rows = []
        for alternative_weight in np.linspace(0, 1, 21):
            blend = (1 - alternative_weight) * legacy_rank + alternative_weight * alternative_rank
            blend_rows.append({
                'legacy_key': legacy_row['key'], 'alternative_key': alternative_row['key'],
                'alternative_weight': float(alternative_weight),
                'prediction_rank_corr': prediction_rank_corr,
                **score_prediction(blend, np.asarray(common['valid']['y']), common['valid']['groups']),
            })
        blend_results = pd.DataFrame(blend_rows).sort_values('selection_score', ascending=False)
        blend_results.to_csv(blend_path, index=False, encoding='utf-8-sig')
        display(blend_results.head(10))
    else:
        print('本次运行未同时包含基准与挑战者预测，保留成绩比较并跳过融合。')
else:
    if not blend_results.empty:
        display(blend_results.sort_values('selection_score', ascending=False).head(10))
    else:
        print('本轮未训练，且没有历史融合结果。')

## 7. 测试集定稿

根据本 Notebook 顶部的显式配置执行定稿训练或跳过不属于本实验的融合步骤。


In [ ]:
def capped_indices_for_split(split, start_time, stop_time, cap):
    row_slice, full_groups = row_slice_for_times(split, start_time, stop_time)
    capped_groups = np.minimum(full_groups, cap).astype(np.int32)
    indices = np.empty(int(capped_groups.sum()), dtype=np.int64)
    source_offset, target_offset = int(row_slice.start), 0
    for full_size, capped_size in zip(full_groups, capped_groups):
        positions = np.linspace(0, int(full_size) - 1, int(capped_size), dtype=np.int64)
        indices[target_offset:target_offset + int(capped_size)] = source_offset + positions
        source_offset += int(full_size); target_offset += int(capped_size)
    assert source_offset == int(row_slice.stop) and target_offset == indices.size
    return indices, capped_groups

def save_atomic_npy(path, array):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(path.suffix + '.partial')
    with partial.open('wb') as handle:
        np.save(handle, array)
    os.replace(partial, path)

if RUN_FINAL_CANDIDATE:
    import lightgbm as lgb
    assert PARAM_PROFILE == 'tuned_public_best'
    assert TRAIN_STOCK_CAP == 1200
    final_window, feature_stop, final_rounds = 1702, 328, 8
    final_start, final_stop = VALID_STOP - final_window, VALID_STOP
    train_indices, train_groups = capped_indices_for_split('train', final_start, TRAIN_STOP, TRAIN_STOCK_CAP)
    valid_indices, valid_groups = capped_indices_for_split('valid', VALID_START, final_stop, TRAIN_STOCK_CAP)
    total_rows = train_indices.size + valid_indices.size
    X_final = np.empty((total_rows, feature_stop), dtype=np.float32)
    y_final = np.empty(total_rows, dtype=np.int8)
    split_at = train_indices.size
    X_final[:split_at] = tree['train'][train_indices, :feature_stop]
    y_final[:split_at] = common['train']['relevance'][train_indices]
    X_final[split_at:] = tree['valid'][valid_indices, :feature_stop]
    y_final[split_at:] = common['valid']['relevance'][valid_indices]
    final_groups = np.concatenate([train_groups, valid_groups]).astype(np.int32)
    assert int(final_groups.sum()) == total_rows == final_window * TRAIN_STOCK_CAP
    del train_indices, valid_indices; gc.collect()
    final_set = lgb.Dataset(X_final, label=y_final, group=final_groups, free_raw_data=True)
    recent_model = lgb.train(LGB_PARAMS, final_set, num_boost_round=final_rounds,
                             callbacks=[lgb.log_evaluation(0)])
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    model_tmp = OUTPUT_DIR / 'model.txt.partial'
    model_tmp.write_text(recent_model.model_to_string(), encoding='utf-8')
    os.replace(model_tmp, OUTPUT_DIR / 'model.txt')
    del final_set, X_final, y_final; gc.collect()

    test_prediction = np.empty(tree['test'].shape[0], dtype=np.float32)
    chunk = 250_000
    for start in range(0, test_prediction.size, chunk):
        stop = min(start + chunk, test_prediction.size)
        test_prediction[start:stop] = recent_model.predict(
            tree['test'][start:stop, :feature_stop], num_iteration=final_rounds
        ).astype(np.float32)
    expert_grid = np.full((442, 5282), 0.5, dtype=np.float32)
    expert_grid[np.asarray(common['test']['time']) - VALID_STOP,
                np.asarray(common['test']['stock'])] = test_prediction
    base_path = PROJECT_ROOT / '04_results' / 'final_submission' / 'prediction.npy'
    base_grid = np.load(base_path)
    assert base_grid.shape == expert_grid.shape == (442, 5282)
    blend_grid = np.full_like(expert_grid, 0.5)
    offset = 0
    for local_time, size in enumerate(common['test']['groups']):
        size = int(size)
        stocks = np.asarray(common['test']['stock'][offset:offset + size], dtype=np.int32)
        base_rank = rankdata(base_grid[local_time, stocks]).astype(np.float32) / size
        expert_rank = rankdata(expert_grid[local_time, stocks]).astype(np.float32) / size
        combined = 0.65 * base_rank + 0.35 * expert_rank
        blend_grid[local_time, stocks] = rankdata(combined).astype(np.float32) / size
        offset += size
    assert offset == test_prediction.size
    candidate_dir = OUTPUT_DIR
    expert_path = candidate_dir / 'expert_prediction.npy'
    blend_path = candidate_dir / 'prediction.npy'
    save_atomic_npy(expert_path, expert_grid)
    save_atomic_npy(blend_path, blend_grid)
    metadata = {
        'status': 'candidate_not_submitted', 'shape': [442, 5282], 'dtype': 'float32',
        'evaluation_count': int(test_prediction.size), 'non_evaluation_value': 0.5,
        'base_weight': 0.65, 'recent1702_weight': 0.35,
        'local_valid_base_rank_ic': 0.092940153549703,
        'local_valid_blend_rank_ic': 0.09444576183929247,
        'base_path': str(base_path), 'expert_path': str(expert_path),
        'blend_path': str(blend_path), 'blend_sha256': sha256(blend_path),
    }
    (candidate_dir / 'metadata.json').write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    print(metadata)
else:
    print('本实验配置不生成近期融合候选。')

## Checks

### 7. 晋级规则与数据契约

In [ ]:
assert manifest['features']['tree_count'] == 419
assert manifest['features']['numeric_count'] == 408
assert manifest['features']['legacy_numeric_prefix'] == 328
assert manifest['features']['tree_categorical_indices'] == list(range(408, 417))
assert int(common['test']['groups'].sum()) == 2_042_538
assert manifest['legacy_compatibility']['status'] == 'passed'

promotion = {'status': 'not_run'}
official_baseline_key = 'tuned_public_best_cap1200__legacy_328_full__r8'
baseline_rows = screening_results[screening_results['key'] == official_baseline_key]
if not baseline_rows.empty and not blend_results.empty:
    baseline = baseline_rows.iloc[0]
    stable = blend_results[
        (blend_results['legacy_key'] == official_baseline_key)
        & (blend_results['mean_rank_ic'] >= baseline['mean_rank_ic'] + 0.001)
        & (blend_results['worst_quarter_rank_ic'] >= baseline['worst_quarter_rank_ic'] - 0.002)
    ].sort_values('selection_score', ascending=False)
    if not stable.empty:
        challenger = stable.iloc[0]
        promotion = {
            'status': 'test_candidate_generated', 'baseline': official_baseline_key,
            'recent_weight': float(challenger['alternative_weight']),
            'mean_delta': float(challenger['mean_rank_ic'] - baseline['mean_rank_ic']),
            'worst_quarter_delta': float(challenger['worst_quarter_rank_ic'] - baseline['worst_quarter_rank_ic']),
        }
print({'data_contract': 'passed', 'promotion': promotion})

## 实验产出

融合结果写入 `04_results/exp_007_recent_window_blend/prediction.npy`，但不会自动覆盖正式提交。


## 8. 保存统一结果


In [ ]:
import hashlib
base_key="tuned_public_best_cap1200__legacy_328_full__r8"; recent_key="tuned_public_best_cap1200__legacy_328_recent1702__r8"
if base_key in screening_predictions and recent_key in screening_predictions:
    base_rank=group_rank_transform(screening_predictions[base_key],common["valid"]["groups"]); recent_rank=group_rank_transform(screening_predictions[recent_key],common["valid"]["groups"]); blend_rank=0.65*base_rank+0.35*recent_rank
    valid_grid=np.full((243,5282),0.5,dtype=np.float32); offset=0
    for local_time,size in enumerate(common["valid"]["groups"]):
        size=int(size); stocks=np.asarray(common["valid"]["stock"][offset:offset+size],dtype=np.int32); valid_grid[local_time,stocks]=blend_rank[offset:offset+size]; offset+=size
    save_atomic_npy(OUTPUT_DIR/"valid_prediction.npy",valid_grid)
prediction_path=OUTPUT_DIR/"prediction.npy"
if not prediction_path.exists(): raise RuntimeError("近期窗口融合没有生成 prediction.npy。")
def file_sha256(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda:handle.read(16*1024*1024),b""): digest.update(block)
    return digest.hexdigest()
metadata_path=OUTPUT_DIR/"metadata.json"; metadata=json.loads(metadata_path.read_text(encoding="utf-8")); metadata.update({"experiment_id":EXPERIMENT_ID,"status":"completed","prediction_sha256":file_sha256(prediction_path),"completed_at":time.strftime("%Y-%m-%d %H:%M:%S")}); tmp=metadata_path.with_suffix(".json.partial"); tmp.write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding="utf-8"); os.replace(tmp,metadata_path)
metrics={"mean_rank_ic":0.09444576183929247,"base_rank_ic":0.092940153549703,"base_weight":0.65,"recent_weight":0.35}; tmp=OUTPUT_DIR/"metrics.json.partial"; tmp.write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding="utf-8"); os.replace(tmp,OUTPUT_DIR/"metrics.json")
report="""# 近期窗口融合实验报告

- 全历史权重：`0.65`
- 近期 1702 期权重：`0.35`
- 验证 RankIC：`0.094446`
- 状态：候选结果，不自动覆盖正式提交。
"""
tmp=OUTPUT_DIR/"experiment_report.md.partial"
tmp.write_text(report,encoding="utf-8")
os.replace(tmp,OUTPUT_DIR/"experiment_report.md")
print("近期窗口融合实验完成：",OUTPUT_DIR)
